# Preprocessing Data:
Take the datasets and preprocess them for model use

# Tasks
1. Handle missing values
2. Convert Categorical fields to numeric (encoding)
3. Normalize numerical features
4. Create train/test split
	- Recent month = test set
	- X months immediately preceding = training set
		- X is not fix and can be a tunable choice 
		- Experiment to find optimal value of X

# Importing data once again

In [20]:
import pandas as pd
import numpy as np

# the root directory of project
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"

# Fetch houses sold in between May 2025 and May 2026
houses_sold_by_month = []

for month in range(0, 13):
    y = "2025" if month < 8 else "2026"
    m =  ""
    if (month + 5 < 10):
        m = f"0{5 + month}"
    elif (month + 5 >= 10 and month + 5 < 13): 
       m = f"{5 + month}" 
    else:
        m = f"0{month - 7}"
    print(f"../raw_data/CRMLSSold{y}{m}.csv")
    filename = f"{root}/raw_data/california/CRMLSSold{y}{m}.csv"
    houses_sold_by_month.append(pd.read_csv(filename))

# Concatenate dataframes
# Ignoring indexes because I'm sure there are duplicates somewhere in the data
houses_sold_2026 = pd.concat(houses_sold_by_month, ignore_index=True)

# Setting float display property with Pandas
pd.set_option('display.float_format', lambda x: '{:,.4f}'.format(x))
pd.set_option('display.max_rows', None)

../raw_data/CRMLSSold202505.csv
../raw_data/CRMLSSold202506.csv
../raw_data/CRMLSSold202507.csv


C:\Users\donutii\AppData\Local\Temp\ipykernel_24020\1573573976.py:21: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  houses_sold_by_month.append(pd.read_csv(filename))


../raw_data/CRMLSSold202508.csv
../raw_data/CRMLSSold202509.csv
../raw_data/CRMLSSold202510.csv
../raw_data/CRMLSSold202511.csv
../raw_data/CRMLSSold202512.csv
../raw_data/CRMLSSold202601.csv


C:\Users\donutii\AppData\Local\Temp\ipykernel_24020\1573573976.py:21: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  houses_sold_by_month.append(pd.read_csv(filename))


../raw_data/CRMLSSold202602.csv
../raw_data/CRMLSSold202603.csv
../raw_data/CRMLSSold202604.csv
../raw_data/CRMLSSold202605.csv


In [21]:
# See columns
print(houses_sold_2026.columns)
print(f'Total size preprocessing: {len(houses_sold_2026)}')

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey',
       'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName',
       'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea',
       'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',
       'ListingId', 'BathroomsTotalInteger', 'City', '

Column Audit: 
- 

# Delete Bad Datapoints / Remove unhelpful columns

In [22]:
# Functions
def print_change(change, size1, size2):
    print(f'{change}: {size1 - size2}, {percent(size1, size2)}% decrease')
def percent(size1, size2):
    return ((size1 - size2) / size1) * 100

In [23]:
# Take out strictly unneeded columns
# Mostly datapoints liable for leakage and other metadata not helpful for analysis (along with information only known after purchase)
houses_sold_2026 = houses_sold_2026.drop(columns=['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'BuyerOfficeName',
                                                  'CoListAgentLastName','BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'ListingKeyNumeric',
                                                  'ContractStatusChangeDate', 'CoBuyerAgentFirstName','PurchaseContractDate', 'ListingContractDate', 'LotSizeDimensions'])

# See columns
print(houses_sold_2026.columns)
print(f'Total size after cutting columns: {len(houses_sold_2026)}')

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'FireplacesTotal',
       'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'MLSAreaMajor',
       'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal',
       'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'BusinessType',
       'StateOrProvince', 'CoveredSpaces', 'MiddleOrJuniorSchool',
       'FireplaceYN', 'Stories', 'HighSchool', 'Leve

In [24]:
# View each column by number of null
null_col_values = houses_sold_2026.isnull().sum()
for col, null_count in null_col_values.items(): 
    total_rows = houses_sold_2026.shape[0]
    pct_null = (null_count / total_rows) * 100
    if pct_null > 90: 
        print(f"Column {col}: {null_count} null values, {pct_null:.2f}% null") 
    # print(f'Column {col}: {null_col_values[col]} null values, {percent(houses_sold_2026[col].size, null_col_values[col].size)}% null')
# print([percent(houses_sold_2026[col].size, null_col_values[col].size) for col in null_col_values.columns])

Column WaterfrontYN: 281665 null values, 99.94% null
Column BasementYN: 277186 null values, 98.35% null
Column FireplacesTotal: 281823 null values, 100.00% null
Column AboveGradeFinishedArea: 281823 null values, 100.00% null
Column TaxAnnualAmount: 280887 null values, 99.67% null
Column BuilderName: 271228 null values, 96.24% null
Column TaxYear: 281661 null values, 99.94% null
Column ElementarySchoolDistrict: 281823 null values, 100.00% null
Column BelowGradeFinishedArea: 280404 null values, 99.50% null
Column BusinessType: 281122 null values, 99.75% null
Column CoveredSpaces: 281823 null values, 100.00% null
Column MiddleOrJuniorSchoolDistrict: 281823 null values, 100.00% null


In [25]:
# Cutting columns that are completely null or have too many null values. I consider the acceptable cutoff to be around 99%, generally
# However, I'm considering leaving in business type (the sample size is so small anyways...)
# WaterfrontYN and BasementYN are easily inferrable. Even if they make up only a small sample of the data I think it's still helpful to have 
# Additionally, BuilderName is not that relevant, and only 96% of listings have it

prev_size = len(houses_sold_2026)
print(f"Previous size: {prev_size}")
print(houses_sold_2026.columns)
houses_sold_2026 = houses_sold_2026.drop(columns=['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 
                                                  'TaxYear', 'ElementarySchoolDistrict',
                                                  'BelowGradeFinishedArea', 'CoveredSpaces', 'MiddleOrJuniorSchoolDistrict'])
print(f"New size: {len(houses_sold_2026)}, -{percent(prev_size, len(houses_sold_2026))}%")

Previous size: 281823
Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'FireplacesTotal',
       'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'MLSAreaMajor',
       'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal',
       'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'BusinessType',
       'StateOrProvince', 'CoveredSpaces', 'MiddleOrJuniorSchool',
       'FireplaceYN', 'Stories

In [26]:
# Dropping duplicate values and entries lacking the target value

prev_size = len(houses_sold_2026)
print(f'Before Dropping Duplicate Values: {prev_size}')
# Drop duplicate values (specifically those with matching unparsed address and close date)
houses_dropped_address_and_date = houses_sold_2026.drop_duplicates(['UnparsedAddress', 'CloseDate'])
print_change('Address and Date Duplicates', houses_sold_2026.size, houses_dropped_address_and_date.size)

# Drop entries that lack closeprice or otherwise lack close price
print_change(f'ClosePrices that are null', houses_dropped_address_and_date.size, houses_dropped_address_and_date.size - houses_dropped_address_and_date.dropna().size)
print_change('Close Prices <= 0', houses_dropped_address_and_date.size, houses_dropped_address_and_date.size - houses_dropped_address_and_date[houses_dropped_address_and_date['ClosePrice']<= 0].size)

# No null prices, so just dropping <= 0 close prices
houses_sold_2026 = houses_dropped_address_and_date.drop(houses_dropped_address_and_date[houses_dropped_address_and_date['ClosePrice']<= 0].index)
print_change(f'Number Removed', prev_size, len(houses_sold_2026))

Before Dropping Duplicate Values: 281823
Address and Date Duplicates: 86867, 0.5815707021783176% decrease
ClosePrices that are null: 0, 0.0% decrease
Close Prices <= 0: 1166, 0.007851982982611427% decrease
Number Removed: 1661, 0.5893770203283621% decrease


In [27]:
# Dealing with impossible lot size 
# Value counts of lot size by property type and subtype
# Naturally, most are condos. Mostly because the shared spaces make it difficult to deal with
    # Not sure how to immute this just yet tho
# For now, I'll only delete the properties that are not condominimums 
values_of_interest = ['PropertyType', 'PropertySubType', 'ClosePrice']
# print(houses_sold_2026[houses_sold_2026['LotSizeSquareFeet'] <= 0][values_of_interest].value_counts())

prev_size = len(houses_sold_2026)
bad_houses = houses_sold_2026[(houses_sold_2026['LotSizeSquareFeet'] <= 0) & (houses_sold_2026['PropertySubType'] != 'Condominium')]
houses_sold_2026 = houses_sold_2026.drop(bad_houses.index)
print_change('Removed rows', prev_size, len(houses_sold_2026))


Removed rows: 1135, 0.40512275040869206% decrease


note: bedroom/bathroom counts inconsistent with lot size also need to be excised. But i'll figure that out later
also need to figure out any other inconsistencies not listed in the document :( (sad face)

# Split Dataset

In [33]:
# Splitting data
# Easy enough for me to do without sklearn, actually
print(f'Total houses sold: {len(houses_sold_2026)}')

testing_set = houses_sold_2026[houses_sold_2026['CloseDate'].str.contains('2026-05')]
print(f'Length of Testing Set: {len(testing_set)}')

training_set = houses_sold_2026.drop(testing_set.index)
print(f'Length of Training Set: {len(training_set)}')

print(f'Checking Length: {len(testing_set) + len(training_set)} = {len(houses_sold_2026)}')

Total houses sold: 279027
Length of Testing Set: 23009
Length of Training Set: 256018
Checking Length: 279027 = 279027


# Deal with Missing Values

In [29]:
# Imputing null values now
# https://medium.com/@prathik.codes/how-to-do-target-encoding-without-data-leakage-the-right-way-280bd24fbc81 
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer

# No shuffling should be done on the data. (I also didn't use random state)
# Currently using 5 splits. Since the population is quite large I might make it bigger
kf = KFold(n_splits=5)